## Рюкзачная криптосистема Шора-Ривеста

In [1]:
from sage.all import *
import numpy as np


# функция, рассчитывающая публичный ключ
def calculate_public_key(p, perm, d, g, t):
    # вычисляем дискретный логарифмы a_i
    a = []
    for i in range(p):
        a.append(discrete_log(t + i, g))
    
    # вычисляем c_i
    c = []
    for i in range(p):
        b = a[perm[i]]
        c.append((b + d) % g.multiplicative_order())
    
    return tuple(c)

# функция, преобразующая число x в вектор y длины p и веса h
def number_to_vector(x, p, h):
    # вычисляем максимальное по величине число, которое может быть
    # зашифровано: 2^(максимальная длина одного сообщения в битах)
    # проверяем, что x меньше максимального числа
    assert x < 2**int(log(binomial(p, h), 2).n())

    y = []

    for k in range(1, p+1):
        bin_coef = binomial(p-k, h)

        if x >= bin_coef:
            y.append(1)
            x = x - bin_coef
            h = h - 1
        else:
            y.append(0)

    return y

# функция зашифрования сообщения
def encrypt(m, c, p, h):
    M = number_to_vector(m, p, h)
    multiplicative_order = p**h - 1

    msg = 0
    for i in range(len(M)):
        if M[i] == 1:
            msg = (msg + c[i] ) % multiplicative_order

    return msg

# функция, преобразующая вектор y длины p и веса h в число x
def vector_to_number(y, p, h):
    x = 0

    for k in range(1, p+1):
        if y[k-1] == 1:
            x = x + binomial(p-k, h)
            h = h - 1

    return x

# функция расшифрования сообщения
def decrypt(msg, p, h, irreducible_poly, g, inverse_perm, d):
    multiplicative_order = p**h - 1
    c_ = (msg - h * d) % multiplicative_order
    u = g ** c_
    # получаем кольцо многочленов над полем GF(p)
    # т.е. коэффициенты многочленов принадлежат полю GF(p)
    R = PolynomialRing(GF(p), "t_")
    # складывает u и irreducible_poly как элементы кольца многочленов
    s = R(u.polynomial()) + R(irreducible_poly)

    M = [0] * p
    # перебираем все множители многочлена s (все линейные)
    for f in factor(s):
        # получаем корень
        coef = 0
        coefs = f[0].coefficients()
        if len(coefs) != 1:
            coef = int(coefs[0])
        
        # восстанавливаем соответствующий бит
        M[inverse_perm[coef]] = 1

    return vector_to_number(M, p, h)

# Генерация параметров криптосистемы

# p - характеристика, h - степень расширения поля
p = 11
h = 5

# определяем символическую переменную
x = var('x')
# неприводимый многочлен (для примера зафиксирован),
# по модулю которого строится поле GF(p^h)
irreducible_polynomial = x**5 + 9*x**3 + 9*x**2 + 3*x + 10

# создаем конечное поле p^h, modulus - неприводимый многочлен
F = GF(p**h, name="t", modulus=irreducible_polynomial)
irreducible_polynomial = F.modulus()
# корень неприводимого многочлена, по модулю
# которого построено поле F
t = F.gen()
# примитивный элемент поля (для примера зафиксирован)
g = 2*t**4 + 8*t**3 + 4*t**2 + 5*t**1 + 2

# случайная перестановка (для примера зафиксирована)
permutation = np.array([5, 6, 7, 2, 8, 10, 4, 0, 1, 3, 9])
# вычисляем обратную перестановку
inverse_permutation = np.arange(p)
inverse_permutation[permutation] = inverse_permutation.copy()

# выбираем случайное число d (для примера зафиксировано)
d = 84997

c = calculate_public_key(p, permutation, d, g, t)

m = 217
print(f"исходное сообщение: {m}")

# Шифрование
msg = encrypt(m, c, p, h)
print(f"зашифрованное сообщение: {msg}")

# Расшифрование
decrypted_msg = decrypt(
    msg, p, h,
    irreducible_polynomial, g,
    inverse_permutation, d,
)
print(f"расшифрованное сообщение: {decrypted_msg}")

исходное сообщение: 217
зашифрованное сообщение: 136807
расшифрованное сообщение: 217


### Генерация параметров



In [41]:
# p - характеристика, h - степень расширения поля
p = 11
h = 5

# определяем символическую переменную
x = var('x')
# неприводимый многочлен (для примера зафиксирован),
# по модулю которого строится поле GF(p^h)
irreducible_polynomial = x**5 + 9*x**3 + 9*x**2 + 3*x + 10

# создаем конечное поле p^h, modulus - неприводимый многочлен
F = GF(p**h, name="t", modulus=irreducible_polynomial)
irreducible_polynomial = F.modulus()
irreducible_polynomial

x^5 + 9*x^3 + 9*x^2 + 3*x + 10

In [42]:
# корень неприводимого многочлена, по модулю
# которого построено поле F
t = F.gen()
# примитивный элемент поля (для примера зафиксирован)
g = 2*t**4 + 8*t**3 + 4*t**2 + 5*t**1 + 2

multiplicative_order = F.order() - 1
multiplicative_order

161050

In [43]:
# проверяем, что элемент g - примитивный
for f, _ in factor(multiplicative_order):
    if g ** (multiplicative_order / f) == 1:
        print('not primitive')
        break

In [44]:
# вычисляем дискретные логарифмы
a = []
for i in range(p):
    a.append(discrete_log(t + i, g))
    print(f"a_{i}: {a[-1]}")

a_0: 134600
a_1: 78964
a_2: 27756
a_3: 28213
a_4: 68089
a_5: 51642
a_6: 38125
a_7: 11604
a_8: 103713
a_9: 123616
a_10: 15120


In [45]:
import numpy as np
# permutation = np.arange(p)
# np.random.shuffle(permutation)

# случайная перестановка (для примера зафиксирована)
permutation = np.array([5, 6, 7, 2, 8, 10, 4, 0, 1, 3, 9])
permutation

array([ 5,  6,  7,  2,  8, 10,  4,  0,  1,  3,  9])

In [46]:
# вычисляем обратную перестановку
inverse_permutation = np.arange(p)
inverse_permutation[permutation] = inverse_permutation.copy()
inverse_permutation

array([ 7,  8,  3,  9,  6,  0,  1,  2,  4, 10,  5])

In [47]:
# from random import randint
# d = randint(0, p**h - 2)

# выбираем случайное число d (для примера зафиксировано)
d = 84997

In [48]:
c = [0] * p

for i in range(len(permutation)):
    c[i] = (a[permutation[i]] + d) % multiplicative_order
    print(f"c_{i} = ({a[permutation[i]]} + {d}) % {multiplicative_order} = {c[i]}")

c_0 = (51642 + 84997) % 161050 = 136639
c_1 = (38125 + 84997) % 161050 = 123122
c_2 = (11604 + 84997) % 161050 = 96601
c_3 = (27756 + 84997) % 161050 = 112753
c_4 = (103713 + 84997) % 161050 = 27660
c_5 = (15120 + 84997) % 161050 = 100117
c_6 = (68089 + 84997) % 161050 = 153086
c_7 = (134600 + 84997) % 161050 = 58547
c_8 = (78964 + 84997) % 161050 = 2911
c_9 = (28213 + 84997) % 161050 = 113210
c_10 = (123616 + 84997) % 161050 = 47563


### Шифрование

In [49]:
bit_num = int(log(binomial(p, h), 2).n())
print(f"Допустимое количество бит в сообщении: {bit_num}")

Допустимое количество бит в сообщении: 8


In [50]:
# возьмем сообщение m
m = 217
# проверяем, что оно удовлетворяет условию на количество бит в сообщении
assert m < 2**bit_num

In [54]:
# представляем наше сообщение в виде бинарного векторв
M = []

message = m
ext = h
for k in range(1, p+1):
    bin_coef = binomial(p-k, ext)

    if message >= bin_coef:
        M.append(1)
        message = message - bin_coef
        ext = ext - 1
    else:
        M.append(0)
        
M

[0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1]

In [55]:
# зашифровываем сообщение

msg = 0
for i in range(len(M)):
    if M[i] == 1:
        msg = (msg + c[i]) % multiplicative_order

msg

136807

### Расшифрование

In [56]:
# вычисляем c'
c_ = (msg - h * d) % multiplicative_order
c_

33922

In [58]:
# вычисляем u
u = g ** c_
u

9*t^4 + 4*t^3 + 7*t^2 + 7*t

In [61]:
# получаем кольцо многочленов над полем GF(p)
# т.е. коэффициенты многочленов принадлежат полю GF(p)
R = PolynomialRing(GF(p), "t_")
# складывает u и irreducible_poly как элементы кольца многочленов
s = R(u.polynomial()) + R(irreducible_polynomial)
s

t_^5 + 9*t_^4 + 2*t_^3 + 5*t_^2 + 10*t_ + 10

In [62]:
# проверяем множители многочлена s
factor(s)

(t_ + 1) * (t_ + 6) * (t_ + 7) * (t_ + 8) * (t_ + 9)

In [64]:
M = [0] * p
# перебираем все множители многочлена s (все линейные)
for f in factor(s):
    # получаем корень
    coef = 0
    coefs = f[0].coefficients()
    if len(coefs) != 1:
        coef = int(coefs[0])

    # восстанавливаем соответствующий бит
    M[inverse_permutation[coef]] = 1

M

[0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1]

In [65]:
# в соответствии с алгоритмом преобразуем вектор обратно в число
x = 0
for k in range(1, p+1):
    if M[k-1] == 1:
        x = x + binomial(p-k, h)
        h = h - 1
       
x

217

### Получили наше исходное сообщение!